# Classificação EEG sem vazamento

Este notebook recebe os epochs já pré-processados, extrai features adequadas para classificadores clássicos e roda **SVM, KNN, Random Forest e XGBoost** com validação por sujeito.

Inclui:
- testes de sanidade;
- checagens explícitas de vazamento;
- teste com rótulos embaralhados;
- avaliação no nível da época e no nível do sujeito.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    HAS_STRATIFIED_GROUP_KFOLD = False

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

SEED = 42
RNG = np.random.default_rng(SEED)

OUT_ROOT = Path(r"C:\Users\tiago\Downloads\Dataset_EEG_Alzheimer\preprocessado_seguros")
MANIFEST_PATH = OUT_ROOT / "manifesto_preprocessado.csv"
EPOCHS_ROOT = OUT_ROOT / "epochs"
RESULTS_DIR = OUT_ROOT / "classificacao"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("XGBoost disponível:", HAS_XGBOOST)
print("StratifiedGroupKFold disponível:", HAS_STRATIFIED_GROUP_KFOLD)

## 1) Carregamento do manifesto e checagens de integridade

In [ ]:
assert MANIFEST_PATH.exists(), f"Manifesto não encontrado: {MANIFEST_PATH}"
assert EPOCHS_ROOT.exists(), f"Pasta de epochs não encontrada: {EPOCHS_ROOT}"

manifesto = pd.read_csv(MANIFEST_PATH)
display(manifesto.head())

required_cols = {
    "sample_id", "grupo", "label", "pais", "arquivo_original", "caminho_original",
    "caminho_saida", "status"
}
missing = required_cols - set(manifesto.columns)
assert not missing, f"Colunas obrigatórias ausentes no manifesto: {missing}"

df_ok = manifesto.query("status == 'OK'").copy()
assert len(df_ok) > 0, "Nenhum sujeito com status OK."

assert df_ok["sample_id"].is_unique, "sample_id deveria ser único por sujeito no manifesto."
assert set(df_ok["label"].unique()).issubset({0, 1}), "Labels fora do esperado."
assert df_ok["caminho_saida"].notna().all(), "caminho_saida com NaN."
assert df_ok["caminho_saida"].apply(lambda x: Path(x).exists()).all(), "Há arquivos .fif faltando."

assert df_ok["caminho_saida"].apply(lambda x: "AD" not in Path(x).name and "HC" not in Path(x).name).all(),     "Vazamento detectado no nome de saída."

print(f"Sujeitos processados com sucesso: {len(df_ok)}")
print(df_ok.groupby(["grupo", "pais"]).size().reset_index(name="N"))

## 2) Extração de features

Cada época vira um vetor fixo para alimentar SVM, KNN, Random Forest e XGBoost.

Features:
- estatísticas no domínio do tempo por canal;
- potência absoluta e relativa por banda por canal;
- estatísticas globais por banda;
- validação de NaN/Inf.

In [ ]:
BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "low_gamma": (30.0, 40.0),
}

EPS = 1e-12

def safe_skew(x):
    mu = x.mean(axis=-1, keepdims=True)
    sd = x.std(axis=-1, keepdims=True) + EPS
    z = (x - mu) / sd
    return np.mean(z**3, axis=-1)

def safe_kurtosis(x):
    mu = x.mean(axis=-1, keepdims=True)
    sd = x.std(axis=-1, keepdims=True) + EPS
    z = (x - mu) / sd
    return np.mean(z**4, axis=-1) - 3.0

def extract_features_from_epochs(epochs, subject_id):
    data = epochs.get_data(copy=True)  # (n_epochs, n_channels, n_times)
    if data.ndim != 3:
        raise ValueError(f"Formato inesperado: {data.shape}")

    ch_names = list(epochs.ch_names)
    n_epochs, n_ch, n_times = data.shape

    feat = {}
    feat["feat_time_mean_global"] = data.mean(axis=(1, 2))
    feat["feat_time_std_global"] = data.std(axis=(1, 2))
    feat["feat_time_rms_global"] = np.sqrt(np.mean(data**2, axis=(1, 2)))
    feat["feat_time_ptp_global"] = data.max(axis=(1, 2)) - data.min(axis=(1, 2))
    feat["feat_time_ll_global"] = np.abs(np.diff(data, axis=2)).sum(axis=(1, 2))
    feat["feat_time_skew_global"] = safe_skew(data.reshape(n_epochs, -1))
    feat["feat_time_kurt_global"] = safe_kurtosis(data.reshape(n_epochs, -1))

    mean_ch = data.mean(axis=2)
    std_ch = data.std(axis=2)
    rms_ch = np.sqrt(np.mean(data**2, axis=2))
    ptp_ch = data.max(axis=2) - data.min(axis=2)
    ll_ch = np.abs(np.diff(data, axis=2)).sum(axis=2)
    skew_ch = safe_skew(data)
    kurt_ch = safe_kurtosis(data)

    for i, ch in enumerate(ch_names):
        feat[f"feat_time_mean__{ch}"] = mean_ch[:, i]
        feat[f"feat_time_std__{ch}"] = std_ch[:, i]
        feat[f"feat_time_rms__{ch}"] = rms_ch[:, i]
        feat[f"feat_time_ptp__{ch}"] = ptp_ch[:, i]
        feat[f"feat_time_ll__{ch}"] = ll_ch[:, i]
        feat[f"feat_time_skew__{ch}"] = skew_ch[:, i]
        feat[f"feat_time_kurt__{ch}"] = kurt_ch[:, i]

    psd = epochs.compute_psd(fmin=0.5, fmax=40.0, verbose=False)
    psd_data, freqs = psd.get_data(return_freqs=True)

    total_power = np.trapz(psd_data, freqs, axis=-1) + EPS
    feat["feat_psd_total_global"] = total_power.mean(axis=1)
    feat["feat_psd_total_std_ch"] = total_power.std(axis=1)

    for band_name, (fmin, fmax) in BANDS.items():
        mask = (freqs >= fmin) & (freqs < fmax)
        if not np.any(mask):
            raise RuntimeError(f"Faixa {band_name} sem frequências no PSD.")

        band_pow = np.trapz(psd_data[:, :, mask], freqs[mask], axis=-1)
        rel_band_pow = band_pow / total_power[:, None]

        feat[f"feat_psd_abs_{band_name}_global_mean"] = band_pow.mean(axis=1)
        feat[f"feat_psd_abs_{band_name}_global_std"] = band_pow.std(axis=1)
        feat[f"feat_psd_rel_{band_name}_global_mean"] = rel_band_pow.mean(axis=1)
        feat[f"feat_psd_rel_{band_name}_global_std"] = rel_band_pow.std(axis=1)

        for i, ch in enumerate(ch_names):
            feat[f"feat_psd_abs_{band_name}__{ch}"] = band_pow[:, i]
            feat[f"feat_psd_rel_{band_name}__{ch}"] = rel_band_pow[:, i]

    df = pd.DataFrame(feat)
    if not np.isfinite(df.to_numpy()).all():
        raise ValueError("Foram encontrados NaN ou Inf nas features.")

    df.insert(0, "sample_id", subject_id)
    df.insert(1, "epoch_idx", np.arange(n_epochs))
    return df

exemplo = df_ok.iloc[0]
epochs_ex = mne.read_epochs(exemplo["caminho_saida"], verbose=False)
df_feat_ex = extract_features_from_epochs(epochs_ex, exemplo["sample_id"])
display(df_feat_ex.head())
print("Shape do exemplo:", df_feat_ex.shape)

## 3) Construção do dataset de classificação

In [ ]:
def build_feature_table(manifest_ok):
    tables = []
    for _, row in manifest_ok.iterrows():
        epochs = mne.read_epochs(row["caminho_saida"], verbose=False)
        df_subj = extract_features_from_epochs(epochs, row["sample_id"])
        df_subj["label"] = int(row["label"])
        df_subj["grupo"] = row["grupo"]
        df_subj["pais"] = row["pais"]
        tables.append(df_subj)
    return pd.concat(tables, ignore_index=True)

df_features = build_feature_table(df_ok)

print("Tabela final de features:", df_features.shape)
display(df_features.head())

metadata_cols = {"sample_id", "epoch_idx", "label", "grupo", "pais"}
feature_cols = [c for c in df_features.columns if c.startswith("feat_")]

assert len(feature_cols) > 0, "Nenhuma feature foi gerada."
assert metadata_cols.isdisjoint(feature_cols), "Coluna de metadado vazando em feature_cols."
assert not df_features[feature_cols].isna().any().any(), "Há NaN nas features."
assert np.isfinite(df_features[feature_cols].to_numpy()).all(), "Há Inf nas features."

print(f"Número de features: {len(feature_cols)}")
print(f"Epochs totais: {len(df_features)}")
print(f"Sujeitos totais: {df_features['sample_id'].nunique()}")

## 4) Funções de avaliação com validação por sujeito

In [ ]:
X = df_features[feature_cols].to_numpy(dtype=float)
y = df_features["label"].to_numpy(dtype=int)
groups = df_features["sample_id"].to_numpy()

subject_labels = df_features.groupby("sample_id")["label"].first().sort_index()
subject_counts = subject_labels.value_counts()
min_subjects_per_class = int(subject_counts.min())

if min_subjects_per_class < 2:
    raise RuntimeError("Poucos sujeitos por classe para validação cruzada com grupos.")

n_splits = min(5, min_subjects_per_class)
if HAS_STRATIFIED_GROUP_KFOLD:
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
else:
    splitter = GroupKFold(n_splits=n_splits)

print("n_splits =", n_splits)
display(subject_counts.rename_axis("classe").reset_index(name="n_sujeitos"))

In [ ]:
def evaluate_grouped_model(name, estimator, X, y, groups, sample_ids, cv):
    fold_rows = []
    oof_prob = np.full(shape=len(y), fill_value=np.nan, dtype=float)
    oof_pred = np.full(shape=len(y), fill_value=np.nan, dtype=float)

    for fold, (tr_idx, te_idx) in enumerate(cv.split(X, y, groups), start=1):
        train_groups = set(groups[tr_idx])
        test_groups = set(groups[te_idx])
        overlap = train_groups & test_groups
        assert len(overlap) == 0, f"Vazamento de sujeito detectado no fold {fold}: {overlap}"

        model = clone(estimator)
        model.fit(X[tr_idx], y[tr_idx])

        if hasattr(model, "predict_proba"):
            prob = model.predict_proba(X[te_idx])[:, 1]
        else:
            decision = model.decision_function(X[te_idx])
            prob = 1 / (1 + np.exp(-decision))

        pred = (prob >= 0.5).astype(int)

        epoch_bac = balanced_accuracy_score(y[te_idx], pred)
        epoch_acc = accuracy_score(y[te_idx], pred)
        epoch_f1 = f1_score(y[te_idx], pred, zero_division=0)
        try:
            epoch_auc = roc_auc_score(y[te_idx], prob)
        except Exception:
            epoch_auc = np.nan

        fold_df = pd.DataFrame({
            "sample_id": sample_ids[te_idx],
            "y_true": y[te_idx],
            "prob": prob,
            "pred": pred,
        })

        subj_df = fold_df.groupby("sample_id", as_index=False).agg(
            y_true=("y_true", "first"),
            prob=("prob", "mean"),
        )
        subj_df["pred"] = (subj_df["prob"] >= 0.5).astype(int)

        subj_bac = balanced_accuracy_score(subj_df["y_true"], subj_df["pred"])
        subj_acc = accuracy_score(subj_df["y_true"], subj_df["pred"])
        subj_f1 = f1_score(subj_df["y_true"], subj_df["pred"], zero_division=0)
        try:
            subj_auc = roc_auc_score(subj_df["y_true"], subj_df["prob"])
        except Exception:
            subj_auc = np.nan

        oof_prob[te_idx] = prob
        oof_pred[te_idx] = pred

        fold_rows.append({
            "model": name,
            "fold": fold,
            "epoch_bac": epoch_bac,
            "epoch_acc": epoch_acc,
            "epoch_f1": epoch_f1,
            "epoch_auc": epoch_auc,
            "subject_bac": subj_bac,
            "subject_acc": subj_acc,
            "subject_f1": subj_f1,
            "subject_auc": subj_auc,
            "n_train_epochs": len(tr_idx),
            "n_test_epochs": len(te_idx),
            "n_train_subjects": len(set(groups[tr_idx])),
            "n_test_subjects": len(set(groups[te_idx])),
        })

    fold_df = pd.DataFrame(fold_rows)
    assert np.isfinite(oof_prob).all(), "OOF prob contém NaN."
    assert np.isfinite(oof_pred).all(), "OOF pred contém NaN."
    return fold_df, oof_prob, oof_pred

def summarize_results(fold_df, label):
    metrics = ["epoch_bac", "epoch_acc", "epoch_f1", "epoch_auc", "subject_bac", "subject_acc", "subject_f1", "subject_auc"]
    out = fold_df[metrics].agg(["mean", "std"]).T
    out.columns = [f"{label}_mean", f"{label}_std"]
    return out

## 5) Modelos

In [ ]:
models = {
    "Dummy_majoritario": DummyClassifier(strategy="most_frequent"),
    "SVM_RBF": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            C=10.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=SEED,
        )),
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(
            n_neighbors=7,
            weights="distance",
            metric="minkowski",
        )),
    ]),
    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        )),
    ]),
}

if HAS_XGBOOST:
    models["XGBoost"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=SEED,
            n_jobs=-1,
        )),
    ])

list(models.keys())

In [ ]:
all_fold_results = []
oof_store = {}

for name, estimator in models.items():
    print(f"\n{'='*80}\nRodando: {name}\n{'='*80}")
    fold_df, oof_prob, oof_pred = evaluate_grouped_model(
        name=name,
        estimator=estimator,
        X=X,
        y=y,
        groups=groups,
        sample_ids=df_features["sample_id"].to_numpy(),
        cv=splitter,
    )
    all_fold_results.append(fold_df)
    oof_store[name] = {"prob": oof_prob, "pred": oof_pred}
    display(fold_df)
    print(fold_df[["epoch_bac", "subject_bac"]].mean())

results = pd.concat(all_fold_results, ignore_index=True)
results_path = RESULTS_DIR / "resultados_cv_modelos.csv"
results.to_csv(results_path, index=False)

print("\nResumo por modelo:")
summary_tables = []
for name in results["model"].unique():
    summary_tables.append(summarize_results(results.query("model == @name"), name))
summary = pd.concat(summary_tables, axis=1)
display(summary)

print(f"Resultados salvos em: {results_path}")